# 07 — Future Value Score (FVS)

**Objective:** Compute the Future Value Score (FVS) for each customer to evaluate forward-looking loyalty engagement beyond historical customer lifetime value (CLV).

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional

from src.config import *
from src.utils import logger, timer, normalize_minmax, save_csv

## 1. FVS Calculation

Calculate normalized FVS components and scale the weighted scores to a standard 0-100 range.

In [2]:
def compute_fvs(
    features_df: pd.DataFrame,
    weights: Optional[dict] = None,
) -> pd.DataFrame:
    if weights is None:
        weights = FVS_WEIGHTS

    df = features_df.copy()

    df["clv_norm"] = normalize_minmax(df["CLV"])
    df["frequency_norm"] = normalize_minmax(df["Flights_12M"])
    df["tier_norm"] = normalize_minmax(df["Loyalty_Card_Ordinal"])
    df["trend_norm"] = normalize_minmax(df["Engagement_Trend"])
    df["redemption_norm"] = normalize_minmax(df["Redemption_Ratio"])
    df["tenure_norm"] = normalize_minmax(df["Tenure_Months"])

    df["FVS_Raw"] = (
        weights["clv_normalized"] * df["clv_norm"]
        + weights["flight_frequency_12m"] * df["frequency_norm"]
        + weights["loyalty_tier"] * df["tier_norm"]
        + weights["engagement_trend"] * df["trend_norm"]
        + weights["redemption_behavior"] * df["redemption_norm"]
        + weights["tenure"] * df["tenure_norm"]
    )

    df["FVS"] = (normalize_minmax(df["FVS_Raw"]) * 100).round(1)

    result = df[[PK, "FVS", "clv_norm", "frequency_norm", "tier_norm",
                 "trend_norm", "redemption_norm", "tenure_norm"]].copy()
    result.columns = [PK, "FVS", "FVS_CLV", "FVS_Frequency", "FVS_Tier",
                      "FVS_Trend", "FVS_Redemption", "FVS_Tenure"]

    logger.info(f"FVS computed for {len(result):,} customers")
    return result

def fvs_segments(fvs_series: pd.Series) -> pd.Series:
    bins = [0, 20, 40, 60, 80, 100]
    labels = ["Very Low", "Low", "Medium", "High", "Very High"]
    return pd.cut(fvs_series, bins=bins, labels=labels, include_lowest=True)

## 2. Compare FVS vs CLV

Analyze where CLV and FVS diverge to identify VIP customers at risk (high CLV, low FVS) and high growth potential members (low CLV, high FVS).

In [3]:
def compare_fvs_vs_clv(features_df: pd.DataFrame, fvs_df: pd.DataFrame) -> pd.DataFrame:
    comparison = features_df[[PK, "CLV", "Churn"]].merge(fvs_df[[PK, "FVS"]], on=PK)
    comparison["CLV_Scaled"] = (normalize_minmax(comparison["CLV"]) * 100).round(1)
    comparison["Divergence"] = (comparison["FVS"] - comparison["CLV_Scaled"]).round(1)

    comparison["Case"] = "Aligned"
    comparison.loc[
        (comparison["CLV_Scaled"] > 60) & (comparison["FVS"] < 40), "Case"
    ] = "High CLV, Low FVS (VIP At Risk)"
    comparison.loc[
        (comparison["CLV_Scaled"] < 40) & (comparison["FVS"] > 60), "Case"
    ] = "Low CLV, High FVS (Growth Potential)"

    correlation = comparison["CLV_Scaled"].corr(comparison["FVS"])
    logger.info(f"CLV vs FVS correlation: {correlation:.3f}")
    return comparison

## 3. Pipeline Execution

Compute FVS scores, run divergence analysis, and save the updated feature dataset.

In [4]:
features_path = FEATURES_DIR / "customer_features.csv"
features_df = pd.read_csv(features_path)

fvs_df = compute_fvs(features_df)
comparison = compare_fvs_vs_clv(features_df, fvs_df)

features_with_fvs = features_df.merge(fvs_df, on=PK)
save_csv(features_with_fvs, FEATURES_DIR / "customer_features_with_fvs.csv")

print(fvs_df["FVS"].describe())

2026-06-12 11:24:27 | airline_loyalty | INFO | FVS computed for 16,737 customers
2026-06-12 11:24:27 | airline_loyalty | INFO | CLV vs FVS correlation: 0.458
2026-06-12 11:24:27 | airline_loyalty | INFO | Saved: customer_features_with_fvs.csv (16,737 rows × 52 cols)


count    16737.000000
mean        36.319896
std         16.873615
min          0.000000
25%         24.500000
50%         35.800000
75%         47.800000
max        100.000000
Name: FVS, dtype: float64
